# 02 · Comparación de modelos

Carga los modelos **ya entrenados** desde `models/<modelo>/` y arma la tabla
comparativa. **No entrena** — es rápido y reproducible.

Compara: mAP@50, mAP@50-95, precision, recall (del `results.csv` de cada modelo)
y FPS de inferencia (medido sobre `data/videos/gol.mp4`). La celda opcional de
mAP por clase (foco en `ball`) requiere tener el dataset descargado.


In [ ]:
from pathlib import Path
import pandas as pd

# Raíz del repo (este notebook vive en notebooks/)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODELS_DIR = ROOT / 'models'

# Modelos a comparar (deben existir como models/<nombre>/)
MODELOS = ['yolo11m', 'yolo26m', 'yolov8m']
MODELOS = [m for m in MODELOS if (MODELS_DIR / m / 'results.csv').exists()]
print('Modelos encontrados:', MODELOS)

## 1. Métricas finales (desde results.csv)


In [ ]:
filas = []
for m in MODELOS:
    df = pd.read_csv(MODELS_DIR / m / 'results.csv')
    df.columns = [c.strip() for c in df.columns]
    last = df.iloc[-1]
    filas.append({
        'modelo': m,
        'mAP50':    round(last['metrics/mAP50(B)'], 4),
        'mAP50-95': round(last['metrics/mAP50-95(B)'], 4),
        'precision':round(last['metrics/precision(B)'], 4),
        'recall':   round(last['metrics/recall(B)'], 4),
    })
tabla = pd.DataFrame(filas).set_index('modelo')
tabla

## 2. FPS de inferencia

Mide la velocidad de cada modelo sobre frames reales de `gol.mp4`. En la Mac
usa `device='mps'`; en Colab, `device=0` (GPU).


In [ ]:
from ultralytics import YOLO
import time, cv2

VIDEO = ROOT / 'data' / 'videos' / 'gol.mp4'
DEVICE = 'mps'      # 'mps' en Mac, 0 en Colab con GPU, 'cpu' si no hay nada
N_FRAMES = 60

# Cargar algunos frames del video
cap = cv2.VideoCapture(str(VIDEO)); frames = []
while len(frames) < N_FRAMES:
    ok, f = cap.read()
    if not ok: break
    frames.append(f)
cap.release()

fps_por_modelo = {}
for m in MODELOS:
    model = YOLO(str(MODELS_DIR / m / 'best.pt'))
    model.predict(frames[0], device=DEVICE, verbose=False)   # warm-up
    t0 = time.perf_counter()
    for f in frames:
        model.predict(f, device=DEVICE, verbose=False)
    dt = time.perf_counter() - t0
    fps_por_modelo[m] = round(len(frames) / dt, 1)
    print(f'{m}: {fps_por_modelo[m]} FPS')

tabla['FPS'] = pd.Series(fps_por_modelo)
tabla

## 3. mAP por clase (opcional — requiere el dataset)

Corre validación sobre el dataset para ver el mAP de cada clase, con foco en
`ball` (la clase difícil). Necesita `FinalV2-1/data.yaml` descargado.


In [ ]:
DATA_YAML = 'FinalV2-1/data.yaml'   # ajustar si el dataset está en otra ruta
import os

if os.path.exists(DATA_YAML):
    por_clase = {}
    for m in MODELOS:
        model = YOLO(str(MODELS_DIR / m / 'best.pt'))
        res = model.val(data=DATA_YAML, verbose=False)
        por_clase[m] = {res.names[i]: round(ap, 4)
                        for i, ap in zip(res.ap_class_index, res.box.ap50)}
    display(pd.DataFrame(por_clase))
else:
    print('Dataset no encontrado en', DATA_YAML, '- saltar esta celda o descargar el dataset.')

## 4. Conclusión

Elegir el modelo ganador balanceando **mAP** (sobre todo en `ball`) y **FPS**.
Anotar acá la decisión y usar ese modelo en el resto del pipeline.
